# Notebook 12: Human validation: bulk microglia, and what to do with single-nucleus data

**Goal.** Test whether the mitophagy-lysosome relationship also weakens with age in people (H4). The dry run uses simulated human data in the same format as GSE99074.

**Cell 1.** Loads the tools and the shared settings. **sys.path.insert** tells Python where the project code lives; **mlc.config** gives every notebook the same folders (PROC = processed data, RES = results tables, FIG = figures) and the profile (*sample* = built-in simulated data, *real* = your downloads). It also tells pandas to print wide tables on one screen. This cell is identical in every notebook.

In [ ]:
import sys; sys.path.insert(0, "../src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from mlc.config import *
from mlc import genesets as G
pd.set_option("display.width", 150); pd.set_option("display.max_columns", 20)
print("profile:", PROFILE, "| cohorts:", cohorts())

**Cell 2.** Maps mouse gene symbols to human ones with the MGI orthology table (some genes map to several, some to none), and loads a human count table (genes x samples) with each person's age and sex. In real data (GSE99074) the count file name and the metadata column names differ: print *counts.columns[:5]* and the GEO sample table and adjust the two paths.

*You should see:* 19 and 44 human genes; (363, 80) genes x samples.

In [ ]:
from mlc.genetics import load_orthologs, to_human
from mlc.analysis import bulk_scores, interaction_test
from mlc import plots
SIM = RAW / "sim"          # REAL DATA: replace with your downloaded files (see the guide)
orth = load_orthologs(SIM / "HOM_MouseHumanSequence_sim.rpt")      # real: RAW / "HOM_MouseHumanSequence.rpt"
H_MITO, H_LYSO = to_human(G.MITOPHAGY, orth), to_human(G.LYSOSOME, orth)
print(len(H_MITO), "human mitophagy genes;", len(H_LYSO), "human lysosome genes")
counts = pd.read_csv(SIM / "human_bulk_counts.csv.gz", index_col=0)
meta = pd.read_csv(SIM / "human_bulk_meta.csv").set_index("sample")
print(counts.shape, "genes x samples; ages", meta.age.min(), "to", meta.age.max())

**Cell 3.** Each person gets a **mitophagy score** and a **lysosome score** (mean of z-scored log2-CPM of the genes). The model is *lyso ~ mito x age + sex*. The row **mito:age** asks whether the mitophagy-lysosome relationship changes with age; a **negative** t-statistic means it weakens, as in mouse. The permutation p shuffles ages 2,000 times so the p-value does not rely on normality.

*You should see:* mito:age t about -2.8; permutation p about 0.03 (the main mito effect is strongly positive).

In [ ]:
d, (na, nb_) = bulk_scores(counts, H_MITO, H_LYSO)
d["age"] = meta.age.reindex(d.index); d["sex_M"] = (meta.sex.reindex(d.index) == "M").astype(float)
f, p, null = interaction_test(d, covars=["sex_M"], n_perm=2000, seed=SEED)
print(f.round(4)); print("permutation p for mito:age interaction =", round(p, 4))

**Cell 4.** **Figure 5.** (a) Each dot is a person; lines show the mitophagy-lysosome slope in the younger and older half: the older half is flatter. (b) The model's t-statistics; the red bar is the interaction.

In [ ]:
plots.fig_human(d, f, p); plt.show()

**Cell 5.** Also records, for every gene, how its expression changes with age in the human samples. Notebook 15 uses this as one evidence layer (does the mouse aging direction agree with human?).

In [ ]:
from mlc.stats import ols_multi
lg = np.log2(counts / counts.sum() * 1e6 + 1); a_ = meta.age.reindex(lg.columns).values
X = np.column_stack([np.ones(len(a_)), (a_ - a_.mean()) / a_.std()])
B, SE, Pv = ols_multi(lg.T.values, X)
pd.DataFrame({"gene": lg.index, "beta_age": B[1], "p": Pv[1]}).to_csv(RES / "human_gene_age.csv", index=False)
print("saved human gene-level age associations:", lg.shape[0], "genes")

**Cell 6.** **Human single-nucleus data.** Use the same notebooks: in 03 call *qc.add_qc(..., species='human')* with *mt_cap=5*; in 04 use human gene symbols (capital letters); in 06 use the human gene lists **H_MITO** and **H_LYSO** from above (and capitalised prefixes RPL/RPS, NDU/SDH/UQCR/COX/ATP5); each person is one sample and has few microglia, so lower MIN_CELLS to 50.

In [ ]:
print("Human single-nucleus cohorts (GSE148822, GSE157827): repeat notebooks 02 to 07 with species='human'")